# Extract Focal Plane Geometry

- author Sylvie Dagoret-Campagne
- creation date 2026-04-02
- LSST pipelines : w_2026_10


Extract CCD geometry (center positions) from LSST camera.
Run this at SLAC and copy the output file back.
- https://github.com/PFLeget/dp2_psf/blob/master/rayTracingFocalPlane/extract_ccd_geometry.py

## Import

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler

import lsst.geom as geom
from lsst.geom import SpherePoint, degrees


from lsst.skymap import PatchInfo, Index2D

from lsst.afw.math import binImage

import lsst.afw.cameraGeom as cameraGeom
import lsst.geom as geom

In [ ]:
import gc

In [ ]:
from astropy.visualization import ZScaleInterval

In [ ]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

In [ ]:
def get_first_existing_dataset(butler, dataset_names, dataId, collections=None):
    for name in dataset_names:
        try:
            obj = butler.get(name, dataId, collections=collections)
            print(f"✔ Found dataset: {name}")
            return obj, name
        except Exception:
            continue

    raise ValueError("No valid dataset found among candidates.")

## Config

In [ ]:
# The output repo is tagged with the Jira ticket number "DM-40356":
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# bad : crash collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'

# bad : collection = "LSSTComCam/runs/DRP/20241101_20241211/w_2024_51/DM-48233"

# not working perhaps because I am using w_2025_10 version
# bad : no ccd visit collection = "LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864"
# bad : no ccd visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_15/DM-50050'
# bad : no cce visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864'
# bad : no cce visit collection collection = 'LSSTComCam/runs/DRP/DP1/w_2025_13/DM-49751'
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
where_clause = "instrument = '" + instrument + "'"
collectionStr = collection[-1].replace("/", "_")
BANDSEL = "r"  # Most fields were observed in red filter

In [ ]:
# Initialize the butler repo:
butler = Butler(repo, collections=collection)
registry = butler.registry

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

In [ ]:
camera = butler.get("camera", collections=collection, instrument=instrument)

In [ ]:
# print(camera.getName(),camera.getNameMap())

## start Focal Plane Extraction

In [ ]:
data = []
for det in camera:
    det_id = det.getId()
    name = det.getName()
    # Get detector center in focal plane coordinates (mm)
    # Using the center pixel transformed to FOCAL_PLANE
    bbox = det.getBBox()
    center_pix_x = (bbox.getMinX() + bbox.getMaxX()) / 2
    center_pix_y = (bbox.getMinY() + bbox.getMaxY()) / 2

    transform = det.getTransform(cameraGeom.PIXELS, cameraGeom.FOCAL_PLANE)
    center_fp = transform.applyForward(geom.Point2D(center_pix_x, center_pix_y))

    # Get corners to determine orientation/size
    corners_pix = [
        geom.Point2D(bbox.getMinX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMaxY()),
        geom.Point2D(bbox.getMinX(), bbox.getMaxY()),
    ]
    corners_fp = [transform.applyForward(c) for c in corners_pix]

    data.append(
        {
            "detector": det_id,
            "name": name,
            "x_center": center_fp.getX(),
            "y_center": center_fp.getY(),
            "corner0_x": corners_fp[0].getX(),
            "corner0_y": corners_fp[0].getY(),
            "corner1_x": corners_fp[1].getX(),
            "corner1_y": corners_fp[1].getY(),
            "corner2_x": corners_fp[2].getX(),
            "corner2_y": corners_fp[2].getY(),
            "corner3_x": corners_fp[3].getX(),
            "corner3_y": corners_fp[3].getY(),
        }
    )

In [ ]:
#!/usr/bin/env python
"""
Extract CCD geometry (center positions) from LSST camera.
Run this at SLAC and copy the output file back.
"""

import numpy as np
from lsst.obs.lsst import LsstCam
import lsst.afw.cameraGeom as cameraGeom
import lsst.geom as geom

camera = LsstCam.getCamera()

# Extract detector info
data = []
for det in camera:
    det_id = det.getId()
    name = det.getName()

    # Get detector center in focal plane coordinates (mm)
    # Using the center pixel transformed to FOCAL_PLANE
    bbox = det.getBBox()
    center_pix_x = (bbox.getMinX() + bbox.getMaxX()) / 2
    center_pix_y = (bbox.getMinY() + bbox.getMaxY()) / 2

    transform = det.getTransform(cameraGeom.PIXELS, cameraGeom.FOCAL_PLANE)
    center_fp = transform.applyForward(geom.Point2D(center_pix_x, center_pix_y))

    # Get corners to determine orientation/size
    corners_pix = [
        geom.Point2D(bbox.getMinX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMaxY()),
        geom.Point2D(bbox.getMinX(), bbox.getMaxY()),
    ]
    corners_fp = [transform.applyForward(c) for c in corners_pix]

    data.append(
        {
            "detector": det_id,
            "name": name,
            "x_center": center_fp.getX(),
            "y_center": center_fp.getY(),
            "corner0_x": corners_fp[0].getX(),
            "corner0_y": corners_fp[0].getY(),
            "corner1_x": corners_fp[1].getX(),
            "corner1_y": corners_fp[1].getY(),
            "corner2_x": corners_fp[2].getX(),
            "corner2_y": corners_fp[2].getY(),
            "corner3_x": corners_fp[3].getX(),
            "corner3_y": corners_fp[3].getY(),
        }
    )

# Save to CSV
import csv

with open("ccd_geometry.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(data)

print(f"Saved {len(data)} detectors to ccd_geometry.csv")
print("Copy this file back to rayTracingFocalPlane/data/")

In [ ]:
data